# Silver Layer: Holidays API CDC

**Source**: `workspace.bronze.holidays_api_cdc`  
**Target**: `workspace.silver.holidays_cdc`  
**Primary Key**: Composite (`country_code`, `date`)

**Transformations:**
1. Normalize country codes (GB → UK)
2. Add computed columns (year, month, day_of_week, day_name)
3. Add is_major_holiday flag
4. Filter to sales data date range (2010-12-29 to 2014-01-28)
5. Rename date column

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, year, month, dayofweek, date_format
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.holidays_api_cdc"
silver_table = "workspace.silver.holidays_cdc"

print("✓ Configuration loaded")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Watermark: {watermark}")
else:
    watermark = "1900-01-01"
    print(f"ℹ️  Initial load")

In [0]:
df = spark.table(bronze_table).filter(col("ingestion_timestamp") > watermark)

new_count = df.count()
print(f"📊 New records: {new_count:,}")

if new_count == 0:
    dbutils.notebook.exit("No new data")

In [0]:
# Normalize country code: GB -> UK to match sales data
df = df.withColumn(
    "country_code",
    when(col("country_code") == "GB", "UK")
    .otherwise(col("country_code"))
)

print("✓ Normalized country codes")

In [0]:
# Filter to sales data date range
df = df.filter(
    (col("date") >= "2010-12-29") & (col("date") <= "2014-01-28")
)

print("✓ Filtered to sales date range")

In [0]:
# Add computed columns for analysis
df = (
    df
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
)

print("✓ Added computed columns")

In [0]:
# Add major holiday flags
df = df.withColumn(
    "is_major_holiday",
    when(col("holiday_name").like("%Christmas%"), True)
    .when(col("holiday_name").like("%New Year%"), True)
    .when(col("holiday_name").like("%Thanksgiving%"), True)
    .when(col("holiday_name").like("%Easter%"), True)
    .otherwise(False)
)

print("✓ Added major holiday flags")

In [0]:
# Rename date column
df = df.withColumnRenamed("date", "holiday_date")

print("✓ Renamed columns")
df.limit(5).display()

In [0]:
if not silver_exists:
    print("Creating Silver table...")
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table))
    print(f"✓ Created: {spark.table(silver_table).count():,} records")
else:
    print("Merging into Silver...")
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("target")
      .merge(
          df.alias("source"),
          "target.country_code = source.country_code AND target.holiday_date = source.holiday_date"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
    print(f"✓ MERGE complete: {spark.table(silver_table).count():,} total")